### vectorless RAG


In [50]:
# --- standard library ---
import os
import random
import time
from pprint import pprint
from typing import Annotated, Any, Dict, List, Literal

# --- third-party ---
import httpx
import wikipedia
from ddgs import DDGS
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

# langchain
from langchain.agents import create_agent
from langchain_core.tools import Tool
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import (
    AIMessage, AnyMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage,
)
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openrouter import ChatOpenRouter
from langchain_tavily import TavilySearch
from langchain_text_splitters import RecursiveCharacterTextSplitter

# langgraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, ToolRuntime, create_react_agent, tools_condition
from langgraph.types import Command


os.environ["GROQ_API_KEY"] = os.environ["GROQ_API_KEY"]
os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGCHAIN_API_KEY"]
os.environ["TAVILY_API_KEY"] = os.environ["TAVILY_API_KEY"]
os.environ["OPEN_ROUTER_API"] = os.environ["OPEN_ROUTER_API"]
os.environ["HF_TOKEN"] = os.environ["HF_TOKEN"]
PAGE_INDEX = os.environ["PAGE_INDEX"]
load_dotenv()

# ToolRuntime.state is typed as dict but our State is a BaseModel -> harmless serializer warning
import warnings
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")




In [51]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [52]:
import json,time
from pageindex import PageIndexClient

pi_client = PageIndexClient(api_key=PAGE_INDEX)


In [53]:
pdf = "project1\\1706.03762v7.pdf"
print(f"📤 Uploading: {pdf}")

result = pi_client.submit_document(pdf)
pdf_id = result["doc_id"] ## id for my document

print(f"✅ Uploaded!")
print(f"📋 Document ID: {pdf_id}")

📤 Uploading: project1\1706.03762v7.pdf
✅ Uploaded!
📋 Document ID: pi-cmvzq8ptq002w0cnn7kkhns2q


C:\Users\Nour_\AppData\Local\Temp\ipykernel_46904\4012553353.py:4: UserWarning: Document "1706.03762v7.pdf" was stored as "1706.03762v7_5.pdf".
  result = pi_client.submit_document(pdf)


In [54]:
print("⏳ Building tree index...")
while True:
    status_result = pi_client.get_document(pdf_id)
    status = status_result.get("status") #.get("status") reads a dict key safely and returns None if the key is missing.
    print(f"   Status: {status}")
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    time.sleep(5) ## makes the program pause for 5 seconds before continuing.



⏳ Building tree index...
   Status: queued
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

In [55]:
tree_result = pi_client.get_tree(pdf_id , node_summary=True)
pageindex_tree = tree_result.get("result",[])
print(f"📊 Top-level sections: {len(pageindex_tree)}")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2)) 

📊 Top-level sections: 1
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 15,
  "summary": "This section introduces the Transformer, a novel sequence modeling architecture that relies entirely on self-attention mechanisms instead of traditional recurrence or convolutions. It covers the authors' affiliations, the abstract and introduction outlining the limitations of recurrent models and the benefits of parallelization, background work, and the detailed model architecture featuring encoder-decoder stacks, multi-head attention, and positional encodings. Furthermore, it compares self-attention to other layer types, details the training regime, presents experimental results and state-of-the-art performance on machine translation and constituency parsing, provides attention visualizations, concludes with future directions, and lists academic references.",
  "text": "",
  "nodes": [
    {
      "title": "Attention Is All You Need (intro)",
    

In [56]:
def print_tree(page_index , indent = 0):
    for index in page_index:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page= index.get("start_index", "?")
        print(f"{prefix}[{index['node_id']}] {index['title']}  (p.{page})")
        if index.get("nodes"):
            print_tree(index["nodes"], indent + 1)

print_tree(pageindex_tree)

[0000] Attention Is All You Need  (p.1)
  └─ [0001] Attention Is All You Need (intro)  (p.1)
  └─ [0002] Abstract  (p.1)
  └─ [0003] 1 Introduction  (p.2)
  └─ [0004] 2 Background  (p.2)
  └─ [0005] 3 Model Architecture  (p.2)
    └─ [0006] 3 Model Architecture (intro)  (p.2)
    └─ [0007] 3.1 Encoder and Decoder Stacks  (p.3)
    └─ [0008] 3.2 Attention  (p.3)
      └─ [0009] 3.2 Attention (intro)  (p.3)
      └─ [0010] 3.2.1 Scaled Dot-Product Attention  (p.4)
      └─ [0011] 3.2.2 Multi-Head Attention  (p.4)
      └─ [0012] 3.2.3 Applications of Attention in our Model  (p.5)
    └─ [0013] 3.3 Position-wise Feed-Forward Networks  (p.5)
    └─ [0014] 3.4 Embeddings and Softmax  (p.5)
    └─ [0015] 3.5 Positional Encoding  (p.6)
  └─ [0016] 4 Why Self-Attention  (p.6)
  └─ [0017] 5 Training  (p.7)
  └─ [0018] 6 Results  (p.8)
    └─ [0019] 6.1 Machine Translation  (p.8)
    └─ [0020] 6.2 Model Variations  (p.8)
    └─ [0021] 6.3 English Constituency Parsing  (p.9)
  └─ [0022] 7 Conclus

In [57]:
def count_nodes(pageindex):
    total = len(pageindex)
    for index in pageindex:
        if index.get("nodes"):
            total+=count_nodes(index["nodes"])
    return total
total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")

🔢 Total nodes in tree: 25


In [58]:
def llm_tree_search(query:str , tree :list , model = llm )->dict:
    def compress(nodes):
        out = [] #This will hold the compressed copies for this level only. 
        for node in nodes:
            entry={
                "node_id": node["node_id"],
                "title" : node["title"],
                "page": node.get("start_index","?"),
                "summary" : node.get("summary","")[:150]
            }
            if node.get("nodes"): #checks for children. Preface has none, so .get returns None, which counts as False, and Python skips the block.
                entry["children"] = compress(node["nodes"])
            out.append(entry)
        return out
    compressed_tree = compress(tree)
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
        Your task: identify which node IDs most likely contain the answer to the query.
        Think step-by-step about which sections are relevant.

        Query: {query}

        Document Tree:
        {json.dumps(compressed_tree, indent=2)}

        Reply ONLY in this exact JSON format:
    {{
    "thinking": "<your step-by-step reasoning>",
    "node_list": ["node_id1", "node_id2"]
    }}"""
    response = model.invoke(
    [{"role": "user", "content": prompt}],
    response_format={"type": "json_object"}
    )   
    return json.loads(response.content)


In [59]:
query = "What is attention ?"
result = llm_tree_search(query, pageindex_tree)
print(result.get("thinking", "N/A"))
print("🎯 Selected Node IDs:", result.get("node_list", []))

The query asks for a definition of attention. In the document tree, the section titled '3.2 Attention' (node_id 0008) explicitly describes attention as a function mapping queries, keys, and values to an output weighted sum. Its child node 0009 ('3.2 Attention (intro)') provides the introductory definition. These nodes are the most relevant for answering the question. Other sections mention attention but do not give the core definition.
🎯 Selected Node IDs: ['0008', '0009']


In [60]:
result

{'thinking': "The query asks for a definition of attention. In the document tree, the section titled '3.2 Attention' (node_id 0008) explicitly describes attention as a function mapping queries, keys, and values to an output weighted sum. Its child node 0009 ('3.2 Attention (intro)') provides the introductory definition. These nodes are the most relevant for answering the question. Other sections mention attention but do not give the core definition.",
 'node_list': ['0008', '0009']}

In [61]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [62]:
def collect_sections(nodes: list) -> list:
    # a parent's "text" is only the part no child holds, so walk each selected node's subtree
    seen, sections = set(), []
    def walk(node):
        if node["node_id"] in seen:
            return
        seen.add(node["node_id"])
        if node.get("text"):
            sections.append(node)
        for child in node.get("nodes", []):
            walk(child)
    for node in nodes:
        walk(node)
    return sections


def generate_answer(query:str , nodes :list , model =llm) -> str:
    sections = collect_sections(nodes)
    if not sections:
        return "No relevant sections were found in the document."

    context = "\n\n---\n\n".join(
        f"[Section: '{node['title']}' | Pages {node.get('start_index', '?')}-{node.get('end_index', '?')}]\n"
        f"{node['text']}"
        for node in sections
    )
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    response = model.invoke([{"role": "user", "content": prompt}])
    return response.content


In [63]:
def vectorless_rag(query:str , tree :list , verbose:bool =True)->str:
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    search_result = llm_tree_search(query , tree)
    node_ids = search_result.get("node_list" , [])

    if verbose :
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    nodes = find_nodes_by_ids(tree , node_ids)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    answer = generate_answer(query, nodes)
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    return answer

    
    

In [64]:
answer = vectorless_rag(
    query="What are the syllabus covered in attention?",
    tree=pageindex_tree
)

🔍 Query: What are the syllabus covered in attention?

🧠 Reasoning: The query asks for the syllabus (topics) covered under the attention portion of the document. We look for nodes whose titles and summaries indicate they describe attention concepts. The main section i...
🎯 Retrieved node IDs: ['0008', '0009', '0010', '0011', '0012']
📄 Sections found: ['3.2 Attention', '3.2 Attention (intro)', '3.2.1 Scaled Dot-Product Attention', '3.2.2 Multi-Head Attention', '3.2.3 Applications of Attention in our Model']

📝 Answer:
The attention chapter covers the following topics:

1. **General definition of an attention function** – mapping a query and a set of key‑value pairs to a weighted‑sum output (Section 3.2 Attention (intro), pp. 3‑4).  
2. **Scaled Dot‑Product Attention** – the specific formulation used (dot‑product of queries and keys, scaling by \(1/\sqrt{d_k}\), softmax, then weighting values) and its relationship to additive attention (Section 3.2.1 Scaled Dot‑Product Attention, pp. 4‑4)

In [65]:
test_queries = [
    "What are the syllabus covered in modern llm finetuning?",
    "What are the syllabus covered in RAG?",
    "Summarize the syllabus of Tokenization Deep Dive?",
]
for q in test_queries:
    ans = vectorless_rag(q, pageindex_tree, verbose=True)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")

🔍 Query: What are the syllabus covered in modern llm finetuning?

🧠 Reasoning: The query asks about the syllabus (topics) covered in modern LLM finetuning. The document is the table of contents of the Transformer paper, which does not explicitly list modern finetuning methods, b...
🎯 Retrieved node IDs: ['0005', '0017']
📄 Sections found: ['3 Model Architecture', '5 Training']

📝 Answer:
The provided material does not contain any information about a syllabus for modern LLM finetuning; it only describes the Transformer architecture, its components, and training details (see sections 3 – 5). Therefore, the requested syllabus cannot be extracted from the given context.
Q: What are the syllabus covered in modern llm finetuning?
A: The provided material does not contain any information about a syllabus for modern LLM finetuning; it only describes the Transformer architecture, its components, and training details (see sections 3 – 5). Therefore, the requested syllabus cannot be extracted from

In [66]:

FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [67]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────
# Krish Naik Academy | 21 Modules | 38 Sections | 481 Topics
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""

In [68]:
def llm_tree_search_with_expert(
        query: str,
        tree:list,
        expert_rules:str,
        model = llm,
)->dict:
        """
        Same as llm_tree_search() but with domain expert rules injected.
        The LLM uses these rules to guide its reasoning.
        """
        def compress(nodes):
                out =[]
                for node in nodes:
                    entry = {"node_id": node["node_id"], "title": node["title"],
                    "page": node.get("start_index", "?"),
                    "summary": node.get("summary", "")[:150]}
                    if node.get("nodes"):
                           entry["children"] = compress(node["nodes"])
                    out.append(entry)
                return out
        prompt = f"""You are a domain expert analyzing a document.
        Find all node IDs that most likely contain the answer to the query.
        Use the expert routing rules below to guide your reasoning.
        
        Query: {query}
        
        Document Tree:
        {json.dumps(compress(tree), indent=2)}
        
        Expert Routing Rules (follow these carefully):
        {expert_rules}
        
        Reply ONLY in this JSON format:
        {{
          "thinking": "<your reasoning, referencing the expert rules>",
          "node_list": ["node_id1", "node_id2"]
        }}"""

        response = model.invoke(
        [{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
        )       
        return json.loads(response.content)

In [71]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Details of the attention?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
start = time.perf_counter()
basic   = llm_tree_search(query, pageindex_tree)
basic_time = time.perf_counter() - start
print("Nodes:", basic.get("node_list"))
print(f"Time: {basic_time:.2f}s")

print()

# With expert rules
print("── With Expert Rules ──")
start = time.perf_counter()
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
guided_time = time.perf_counter() - start
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])
print(f"Time: {guided_time:.2f}s")

print()
print(f"⏱️ Difference (expert - basic): {guided_time - basic_time:+.2f}s")

🔍 Query: Details of the attention?

── Without Expert Rules ──
Nodes: ['0008', '0010', '0011', '0012']
Time: 1.40s

── With Expert Rules ──
Nodes: ['0008', '0009', '0010', '0011', '0012', '0016']
Reasoning: The query asks for details of the attention mechanism. According to the expert routing rules, attention falls under module M3 (Transformers 101). In the document, the sections that describe attention are under the '3.2 Attention' node and its children, which cover the general concept, scaled dot-pro
Time: 16.88s

⏱️ Difference (expert - basic): +15.48s


In [74]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

start = time.time()
# Run it
answer = expert_rag(
    query="Details of the syllabus of key , value",
    tree=pageindex_tree,
    rules=FINANCIAL_EXPERT_RULES
)
end = time.time()
print(answer)
print(end-start)

- **What a key and a value are** – In the attention function a *key* and a *value* are each a vector that belongs to a key‑value pair.  The query is compared to each key to produce a weight, and the weighted sum is taken over the corresponding values (Section ‘3.2 Attention (intro)’, pp. 3‑4).  

- **Dimensionality** –  
  * Keys have dimension \(d_k\) and values have dimension \(d_v\) (Section ‘3.2.1 Scaled Dot‑Product Attention’, p. 4).  
  * In the multi‑head variant each head linearly projects the original vectors to \(d_k\) (for keys) and \(d_v\) (for values) (Section ‘3.2.2 Multi‑Head Attention’, pp. 4‑5).  

- **Concrete numbers used in the model** – The paper uses \(h = 8\) parallel heads and sets  
  \[
  d_k = d_v = \frac{d_{\text{model}}}{h} = 64,
  \] 
  because \(d_{\text{model}} = 512\) (Section ‘3.2.2 Multi‑Head Attention’, p. 5).  

- **Projection matrices** – For each head \(i\) the keys and values are obtained by learned linear maps  
  \[
  K_i = K\,W_i^{K},\qquad V_